## Read daata from csv datasets

In [14]:
import csv

def build_table(file_path,length=5):
    table = ""
    with open(file_path, 'r') as f:
        reader = csv.reader(f)
        for i, row in enumerate(reader):
            if i < length :
                table += "\t".join(row) + "\n"
    return table

In [15]:
finance_data = {}
hospital_data = {}
sales_data = {}

finance_data['table'] = build_table('datasets/financials.csv',7)
hospital_data['table'] = build_table('datasets/hospital_readmissions.csv',7)
sales_data['table'] = build_table('datasets/sales_data.csv',7)

finance_data['table_page_title'] = "Financial Sample Data"
hospital_data['table_page_title'] = "Hospital Readmission Sample Data"
sales_data['table_page_title'] = "Sales Sample Data"

finance_data['table_section_title'] = "Finance Data"
hospital_data['table_section_title'] ="Hospital Readmission Data"
sales_data['table_section_title'] = "Sales Data"

finance_data['table_section_text'] = "This is a dataset that requires a lot of preprocessing with amazing EDA insights for a company. A dataset consisting of sales and profit data sorted by market segment and country/region."
sales_data['table_section_text'] = "This dataset represents synthetic sales data generated for practice purposes only. It is not real-time or based on actual business operations, and should be used solely for educational or testing purposes. The dataset contains information that simulates sales transactions across different products, regions, and customers. Each row represents an individual sale event with various details associated with it."

finance_data['highlighted_cells'] = [(3, 6)]   
hospital_data['highlighted_cells'] = [(4,2)] 
sales_data['highlighted_cells'] = [(2, 3)] 

print("Finance Data Table:\n", finance_data['table'])
print("Hospital Data Table:\n", hospital_data['table'])
print("Sales Data Table:\n", sales_data['table'])


Finance Data Table:
 Segment	Country	 Product 	 Discount Band 	 Units Sold 	 Manufacturing Price 	 Sale Price 	 Gross Sales 	 Discounts 	  Sales 	 COGS 	 Profit 	Date	Month Number	 Month Name 	Year
Government	Canada	 Carretera 	 None 	 $1,618.50 	 $3.00 	 $20.00 	 $32,370.00 	 $-   	 $32,370.00 	 $16,185.00 	 $16,185.00 	01/01/2014	1	 January 	2014
Government	Germany	 Carretera 	 None 	 $1,321.00 	 $3.00 	 $20.00 	 $26,420.00 	 $-   	 $26,420.00 	 $13,210.00 	 $13,210.00 	01/01/2014	1	 January 	2014
Midmarket	France	 Carretera 	 None 	 $2,178.00 	 $3.00 	 $15.00 	 $32,670.00 	 $-   	 $32,670.00 	 $21,780.00 	 $10,890.00 	01/06/2014	6	 June 	2014
Midmarket	Germany	 Carretera 	 None 	 $888.00 	 $3.00 	 $15.00 	 $13,320.00 	 $-   	 $13,320.00 	 $8,880.00 	 $4,440.00 	01/06/2014	6	 June 	2014
Midmarket	Mexico	 Carretera 	 None 	 $2,470.00 	 $3.00 	 $15.00 	 $37,050.00 	 $-   	 $37,050.00 	 $24,700.00 	 $12,350.00 	01/06/2014	6	 June 	2014
Government	Germany	 Carretera 	 None 	 $1,513.00 	 

In [16]:
# Dump the data to JSON files
import json

with open('datasets/finance_data.json', 'w') as f:
    json.dump(finance_data, f, indent=4)
with open('datasets/hospital_data.json', 'w') as f:
    json.dump(hospital_data, f, indent=4)
with open('datasets/sales_data.json', 'w') as f:
    json.dump(sales_data, f, indent=4)

In [17]:
# Load json datasets from files
import json
finance_dataset = json.load(open('datasets/finance_data.json'))
hospital_dataset = json.load(open('datasets/hospital_data.json'))
sales_dataset = json.load(open('datasets/sales_data.json'))

In [18]:
# display dataset samples
print("Finance Dataset Sample:\n", finance_dataset)
print("Hospital Dataset Sample:\n", hospital_dataset)    
print("Sales Dataset Sample:\n", sales_dataset)

Finance Dataset Sample:
 {'table': 'Segment\tCountry\t Product \t Discount Band \t Units Sold \t Manufacturing Price \t Sale Price \t Gross Sales \t Discounts \t  Sales \t COGS \t Profit \tDate\tMonth Number\t Month Name \tYear\nGovernment\tCanada\t Carretera \t None \t $1,618.50 \t $3.00 \t $20.00 \t $32,370.00 \t $-   \t $32,370.00 \t $16,185.00 \t $16,185.00 \t01/01/2014\t1\t January \t2014\nGovernment\tGermany\t Carretera \t None \t $1,321.00 \t $3.00 \t $20.00 \t $26,420.00 \t $-   \t $26,420.00 \t $13,210.00 \t $13,210.00 \t01/01/2014\t1\t January \t2014\nMidmarket\tFrance\t Carretera \t None \t $2,178.00 \t $3.00 \t $15.00 \t $32,670.00 \t $-   \t $32,670.00 \t $21,780.00 \t $10,890.00 \t01/06/2014\t6\t June \t2014\nMidmarket\tGermany\t Carretera \t None \t $888.00 \t $3.00 \t $15.00 \t $13,320.00 \t $-   \t $13,320.00 \t $8,880.00 \t $4,440.00 \t01/06/2014\t6\t June \t2014\nMidmarket\tMexico\t Carretera \t None \t $2,470.00 \t $3.00 \t $15.00 \t $37,050.00 \t $-   \t $37,050.00

## Prompt Construction from data


In [6]:
def coerce_table(table):
    """Convert raw string tables from JSON into row/cell objects expected by the prompt builder."""
    if isinstance(table, str):
        rows = []
        for line in table.splitlines():
            line = line.strip()
            if not line:
                continue
            cells = [{"value": cell.strip(), "is_header": False} for cell in line.split("\t")]
            rows.append(cells)
        if rows:
            for cell in rows[0]:
                cell["is_header"] = True
        return rows
    return table


def get_header_block_end(table):
    """The header block is the maximal prefix of rows where EVERY cell is
    is_header=True. This correctly excludes tables where only the first
    COLUMN is marked as a row-label inside data rows (e.g. Chicago Bears'
    'Year' column) - those rows are only partially header-marked, so they
    fail the 'every cell' test and the block stops before them."""
    table = coerce_table(table)
    end = 0
    for row in table:
        if row and all(cell.get("is_header") for cell in row):
            end += 1
        else:
            break
    return end


def expand_header_row(row):
    """Expands a header row's cells across their column_span so position i
    lines up with actual column i, matching data-row layout."""
    expanded = []
    for cell in row:
        expanded.extend([cell["value"]] * max(cell.get("column_span", 1), 1))
    return expanded


def is_locally_uniform(table, header_block_end, target_row_idx, header_width):
    """Only require rows BETWEEN the header and the target to match the
    header's width. A width change AFTER the target row (e.g. a second
    inline sub-table starting later) doesn't make the target's own column
    position untrustworthy."""
    table = coerce_table(table)
    for row in table[header_block_end:target_row_idx + 1]:
        if len(row) != header_width:
            return False
    return True


def get_column_header(table, target_row_idx, target_col_idx):
    table = coerce_table(table)
    header_block_end = get_header_block_end(table)
    if header_block_end == 0 or target_row_idx < header_block_end:
        return ""
    if not is_locally_uniform(table, header_block_end, target_row_idx, len(table[header_block_end - 1])):
        return ""  # can't trust positional alignment in this table - skip, don't guess

    parts = []
    for row in table[:header_block_end]:
        expanded = expand_header_row(row)
        if target_col_idx >= len(expanded):
            return ""
        val = expanded[target_col_idx].strip()
        if val and (not parts or parts[-1] != val):
            parts.append(val)
    return " ".join(parts)


WINDOW = 3  # rows of context kept on each side of a highlighted row


def select_rows(table, highlighted_cells, window=WINDOW):
    """Returns a sorted list of row indices to keep: the header block, plus
    a window of rows around every highlighted row."""
    table = coerce_table(table)
    header_end = get_header_block_end(table)
    keep = set(range(header_end))
    for r, _ in highlighted_cells:
        keep.update(range(max(0, r - window), min(len(table), r + window + 1)))
    return sorted(keep)


def build_table_str(table, keep_indices, highlighted_cells):
    """Serializes only the kept rows. Each row keeps a cheap [i] row-index
    prefix (proven sufficient for row lookup on its own - see Section 4's
    intro). Only the specific highlighted cell(s) get the heavier [R{row}C{col}]
    tag, wrapped around their value in place - not every cell in the row.

    This is a deliberate revision: tagging EVERY cell (the first version of
    this fix) roughly doubled average token length, because T5's tokenizer
    splits mixed letter+digit+bracket patterns like "[R123C45]" into many
    small tokens, and tagging every column of every row scales that cost by
    (rows x columns) instead of just (rows). Tagging only the highlighted
    cells keeps the same literal-lookup benefit where it's actually needed,
    at a cost of roughly (rows) + (a handful of highlighted cells), not
    (rows x columns)."""
    table = coerce_table(table)
    highlighted_set = {(r, c) for r, c in highlighted_cells}
    lines = []
    prev = None
    for i in keep_indices:
        if prev is not None and i != prev + 1:
            lines.append("...")
        cell_strs = []
        for j, cell in enumerate(table[i]):
            if (i, j) in highlighted_set:
                cell_strs.append(f"({i},{j}){cell['value']}")
            else:
                cell_strs.append(cell["value"])
        lines.append(f"[{i}] " + " | ".join(cell_strs))
        prev = i
    return "\n".join(lines)


In [8]:
def build_prompt(sample):
    """ Builds a windowed prompt for the model, keeping only the rows needed
    to describe the highlighted cells. Highlighted cells are tagged with
    [R{row}C{col}] both in the table body and in the Highlighted Cells
    section - the same literal anchor in both places, so the model can look
    up the cell directly instead of resolving "Row: X, Column: Y" by
    counting. Only highlighted cells are tagged (not every cell) to keep
    token cost close to the un-tagged windowed baseline - see Section 4's
    build_table_str docstring for why full-table tagging was rejected. """
    table = sample["table"]
    highlighted_cells = sample["highlighted_cells"]

    keep_indices = select_rows(table, highlighted_cells)
    table_str = build_table_str(table, keep_indices, highlighted_cells)

    highlighted_lines = []
    for row, col in highlighted_cells:
        header = get_column_header(table, row, col)
        label = f" ({header})" if header else ""
        highlighted_lines.append(f"({row},{col}){label}")
    highlighted_str = "\n".join(highlighted_lines)

    prompt = f"""Task:
Generate a single factual sentence describing the information contained in the highlighted cells.
Use only information provided below.
Do not invent facts.\n"""
    if "table_page_title" in sample and sample["table_page_title"]:
        prompt += f"Page Title:\n{sample['table_page_title']}\n"
    if "table_section_title" in sample and sample["table_section_title"]:
        prompt += f"Section Title:\n{sample['table_section_title']}\n"
    if "table_section_text" in sample and sample["table_section_text"]:
        prompt += f"Additional Context:\n{sample['table_section_text']}\n"
    if highlighted_cells:
        prompt += f"Highlighted Cells:\n{highlighted_str}\n"

    prompt += f"Table:\n{table_str}\nAnswer:\n"
    return prompt


In [19]:
finance_prompt = build_prompt(finance_dataset)
hospital_prompt = build_prompt(hospital_dataset)
sales_prompt = build_prompt(sales_dataset)

print("Finance Prompt:\n", finance_prompt)
print("Hospital Prompt:\n", hospital_prompt)
print("Sales Prompt:\n", sales_prompt)

Finance Prompt:
 Task:
Generate a single factual sentence describing the information contained in the highlighted cells.
Use only information provided below.
Do not invent facts.
Page Title:
Financial Sample Data
Section Title:
Finance Data
Additional Context:
This is a dataset that requires a lot of preprocessing with amazing EDA insights for a company. A dataset consisting of sales and profit data sorted by market segment and country/region.
Highlighted Cells:
(3,6) (Sale Price)
Table:
[0] Segment | Country | Product | Discount Band | Units Sold | Manufacturing Price | Sale Price | Gross Sales | Discounts | Sales | COGS | Profit | Date | Month Number | Month Name | Year
[1] Government | Canada | Carretera | None | $1,618.50 | $3.00 | $20.00 | $32,370.00 | $- | $32,370.00 | $16,185.00 | $16,185.00 | 01/01/2014 | 1 | January | 2014
[2] Government | Germany | Carretera | None | $1,321.00 | $3.00 | $20.00 | $26,420.00 | $- | $26,420.00 | $13,210.00 | $13,210.00 | 01/01/2014 | 1 | January

## Loading LoRA fine tuned model (Base Model + LoRA Adapter)


In [27]:
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

base_model = AutoModelForSeq2SeqLM.from_pretrained("./models/flan-t5-base-local", device_map="auto", trust_remote_code=True, dtype=torch.bfloat16)

fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    "./models/flan-t5-base-totto-lora-finetuned-optimized"
)


tokenizer = AutoTokenizer.from_pretrained("./models/flan-t5-base-local")

fine_tuned_model.eval()
fine_tuned_model.cuda()



Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1214.41it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): T5ForConditionalGeneration(
      (shared): Embedding(32128, 768)
      (encoder): T5Stack(
        (embed_tokens): Embedding(32128, 768)
        (block): ModuleList(
          (0): T5Block(
            (layer): ModuleList(
              (0): T5LayerSelfAttention(
                (SelfAttention): T5Attention(
                  (q): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=768, out_features=8, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=8, out_features=768, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
               

In [ ]:
def generate_summary(model, tokenizer, prompt, max_new_tokens=64, num_beams=4):
    inputs = tokenizer(
        prompt,
        truncation=True,
        max_length=1024,          
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            do_sample=False,      # matches quantitative eval settings
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [28]:
for name, prompt_text in [
    ("Finance", finance_prompt),
    ("Hospital", hospital_prompt),
    ("Sales", sales_prompt),
]:
    summary_1 = generate_summary(fine_tuned_model, tokenizer, prompt_text)
    print(f"--- {name} ---")
    print(summary_1)
    print()


--- Finance ---
Midmarket sold 15.00 units for $15.00.

--- Hospital ---
The 36-year-old was readmitted to the hospital.

--- Sales ---
West-Bob sold a product in the West-Bob region.

